In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import pickle
from tqdm import tqdm
from torchvision import transforms

# Distillation Dataset

In [2]:
train_df1_path = '/kaggle/input/datasets/shahdammar/distillationdataset/biovil_t_fixed_full.pkl'
train_df2_path = '/kaggle/input/datasets/shahdammar/distillationdataset/biovil_t_fixed_full2.pkl'
val_df_path = "/kaggle/input/datasets/shahdammar/distillationdataset/biovil_t_validation_full.pkl"

## Exploring Dataset 

In [3]:
with open(train_df1_path, 'rb') as f:
    data = pickle.load(f)

print(f"Data type: {type(data)}")

Data type: <class 'pandas.core.frame.DataFrame'>


In [4]:
data.head()

,subject_id,study_id,report_text,image_paths,num_views_used,raw_report_text,source_row_index,report_embedding,image_embedding,matched_cosine,random_negative_cosine,cosine_margin_vs_random
0,10000032,50414267,"Findings: There is no focal consolidation, ple...",[/kaggle/input/datasets/simhadrisadaram/mimic-...,1,"Findings: There is no focal consolidation, ple...",0,"[0.0410073, -0.02050806, -0.25146016, 0.098930...","[0.09292349, 0.027857743, -0.19782346, 0.01236...",0.654669,-0.435985,1.090655
1,10000032,53189527,"Findings: The cardiac, mediastinal and hilar c...",[/kaggle/input/datasets/simhadrisadaram/mimic-...,1,"Findings: The cardiac, mediastinal and hilar c...",0,"[3.5338595e-05, -0.048801687, -0.122308925, 0....","[0.06965752, -0.0089393165, -0.21145461, 0.040...",0.377239,-0.123714,0.500953
2,10000032,53911762,Findings: Single frontal view of the chest pro...,[/kaggle/input/datasets/simhadrisadaram/mimic-...,2,Findings: Single frontal view of the chest pro...,0,"[-0.013153604, 0.039516266, -0.2214446, 0.0931...","[0.040271934, 0.023725748, -0.33616465, 0.1283...",0.732569,-0.367367,1.099936
3,10000032,56699142,Findings: The lungs are clear of focal consoli...,[/kaggle/input/datasets/simhadrisadaram/mimic-...,1,Findings: The lungs are clear of focal consoli...,0,"[0.013930174, -0.053979196, -0.15940279, 0.055...","[0.0858494, -0.008497886, -0.23187724, 0.03901...",0.478240,-0.218972,0.697212
4,10000764,57375967,Findings: PA and lateral views of the chest pr...,[/kaggle/input/datasets/simhadrisadaram/mimic-...,1,Findings: PA and lateral views of the chest pr...,1,"[0.12738413, 0.08956883, -0.02368909, -0.01456...","[0.09522368, 0.090368696, -0.20908515, 0.04757...",0.632557,0.054910,0.577647


In [5]:
data.columns

Index(['subject_id', 'study_id', 'report_text', 'image_paths',
       'num_views_used', 'raw_report_text', 'source_row_index',
       'report_embedding', 'image_embedding', 'matched_cosine',
       'random_negative_cosine', 'cosine_margin_vs_random'],
      dtype='object')

In [6]:
with open(val_df_path, 'rb') as f:
    val_df = pickle.load(f)

print(len(val_df))

1201


## Merge Training Dataset

In [7]:
with open(train_df1_path, 'rb') as f:
    df1 = pickle.load(f)
    
with open(train_df2_path, 'rb') as f:
    df2 = pickle.load(f)
    
train_df = pd.concat([df1, df2], ignore_index=True)

In [8]:
len(train_df)

142942

In [9]:
train_df['num_views_used'].value_counts()

num_views_used
1     127511
2      14613
3        692
4         98
5         19
6          6
7          1
8          1
10         1
Name: count, dtype: int64

## Dataset Class  
<font size='4'>Some studies include multiple views. The maximum number of views to process is set to 3, as most of the dataset has 3 views or fewer.</font>

In [10]:
class DistillationDataset(Dataset):
    
    def __init__(self, dataframe, transform=None, max_views=3):
        self.df = dataframe
        self.max_views = max_views
        if transform is not None:
            self.transform = transform
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),  # Standard for MobileViT
                transforms.ToTensor(),         
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], # BioViL defaults
                    std=[0.229, 0.224, 0.225]
                )
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paths = row['image_paths']
        
        # Determine how many views to actually process
        actual_num_views = len(paths)
        num_to_process = min(actual_num_views, self.max_views)
        
        view_tensors = []
        for i in range(self.max_views):
            if i < num_to_process:
                # Load actual image
                try:
                    img = Image.open(paths[i]).convert('RGB')
                    if self.transform:
                        img = self.transform(img)
                    view_tensors.append(img)
                except:
                    view_tensors.append(torch.zeros((3, 224, 224)))
            else:
                # Padding: Add a zero tensor for the "empty" views
                view_tensors.append(torch.zeros((3, 224, 224)))
        
        # [Max_Views, 3, 224, 224]
        stacked_images = torch.stack(view_tensors)
        teacher_emb = torch.tensor(row['image_embedding'], dtype=torch.float32)
        
        # We return the actual count so we don't average the zero-padding
        return stacked_images, teacher_emb, num_to_process

## Data Loaders

In [11]:
train_ds = DistillationDataset(train_df)
val_ds = DistillationDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

# Student

<font size='4'> **Backbone:** MobileViT-Small provides a hybrid CNN-Transformer architecture for global and local feature awareness.  
**Projection Head:** A multi-layer mapper that aligns the 640-dimensional latent features with the 128-dimensional BioViL teacher space.  
The model is designed to handle a variable number of X-ray views per study by processing images individually and performing Late Fusion (averaging).  </font>

In [12]:
class MobileViTStudent(nn.Module):
    
    def __init__(self, teacher_dim=128):
        super().__init__()
        self.backbone = timm.create_model('mobilevit_s', pretrained=True, num_classes=0)
        self.mapper = nn.Sequential(
            nn.Linear(640, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, teacher_dim)
        )
        

    def forward(self, x, counts):
        """
        x: [Batch, Max_Views, 3, 224, 224]
        counts: [Batch] -> contains how many real images are in each study
        """
        batch_size, max_views, c, h, w = x.shape
        
        # 1. Flatten to process every image across the whole batch
        x = x.view(-1, c, h, w)        # [Batch * Max_Views, 3, 224, 224]
        
        # 2. Extract per-image features
        features = self.backbone(x)    # [Batch * Max_Views, 640]
        per_view_embs = self.mapper(features)   # [Batch * Max_Views, 128]
        
        # 3. Reshape back to [Batch, Max_Views, 128]
        per_view_embs = per_view_embs.view(batch_size, max_views, -1)
        
        # 4. --- MATCHING TEACHER LOGIC ---
        # Instead of a simple .mean(), we sum only the valid images and divide by the count
        # This ignores the zero-padded tensors
        
        # Create a mask of real images (1 for real, 0 for padded)
        # mask shape: [Batch, Max_Views, 1]
        mask = torch.arange(max_views).expand(batch_size, max_views).to(x.device)
        mask = (mask < counts.unsqueeze(1)).float().unsqueeze(-1)
        
        # Apply mask and sum
        sum_embs = torch.sum(per_view_embs * mask, dim=1)   # [Batch, 128]
        
        # Calculate mean: Divide by actual counts
        # (counts.view(-1, 1) ensures we divide each study by its own number of images)
        mean_emb = sum_embs / counts.view(-1, 1).float()
        
        # 5. Normalize like F.normalize in the teacher's code
        final_emb = F.normalize(mean_emb, p=2, dim=-1)
        
        return final_emb

# Student Training
<font size='4'> Feature distillation of BioViL image embeddings via the DistillationDataset using an **embedding regression** strategy. </font>

In [13]:
def distillation_loss(student_emb, teacher_emb):
    """
    Option A: Regression Loss
    Combines MSE (Value match) and Cosine (Direction match)
    """
    mse = nn.MSELoss()(student_emb, teacher_emb)
    cos_loss = 1 - F.cosine_similarity(student_emb, teacher_emb).mean()
    return mse + cos_loss

In [14]:
def train_one_epoch(model, loader, optimizer, device, scaler):
    model.train()
    total_loss = 0
    
    pbar = tqdm(loader, desc="Training", leave=False)
    for imgs, targets, counts in pbar:
        imgs, targets, counts = imgs.to(device), targets.to(device), counts.to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            preds = model(imgs, counts)
            loss = distillation_loss(preds, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
        
    return total_loss / len(loader)

In [15]:
def validate(model, loader, device):
    model.eval()
    total_cos_sim = 0
    total_mse = 0
    
    with torch.no_grad():
        for imgs, targets, counts in tqdm(loader, desc="Validating", leave=False):
            imgs, targets, counts = imgs.to(device), targets.to(device), counts.to(device)
            
            preds = model(imgs, counts)
            
            # Calculate metrics
            cos_sim = F.cosine_similarity(preds, targets).mean()
            mse = F.mse_loss(preds, targets)
            
            total_cos_sim += cos_sim.item()
            total_mse += mse.item()
            
    avg_cos = total_cos_sim / len(loader)
    avg_mse = total_mse / len(loader)
    return avg_cos, avg_mse

In [16]:
def run_distillation(model, train_loader, val_loader, optimizer, epochs, device, save_name="best_student.pth"):
    print(f"Starting Distillation on {device}...")
    scaler = torch.cuda.amp.GradScaler()
    best_val_cos = -1.0  # Initialize with worst possible similarity
    
    history = {"train_loss": [], "val_cos": [], "val_mse": []}

    for epoch in range(epochs):
        # Train
        avg_train_loss = train_one_epoch(model, train_loader, optimizer, device, scaler)
        
        # Validate
        val_cos, val_mse = validate(model, val_loader, device)
        
        # Store History
        history["train_loss"].append(avg_train_loss)
        history["val_cos"].append(val_cos)
        history["val_mse"].append(val_mse)

        # Print Summary
        print(f"\n--- Epoch {epoch+1}/{epochs} ---")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Cosine Similarity: {val_cos:.4f} (Target: 1.0)")
        print(f"Val MSE: {val_mse:.4f}")

        # Save Best Model (Based on Cosine Similarity)
        if val_cos > best_val_cos:
            best_val_cos = val_cos
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_cos': val_cos,
            }, save_name)
            print(f"⭐ New Best Model Saved! (Similarity: {val_cos:.4f})")
            
    print("\nTraining Complete!")
    return history

In [17]:
device = torch.device("cuda")
model = MobileViTStudent().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

model.safetensors:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

In [18]:
run_distillation(model, train_loader, val_loader, optimizer, 10, device)

/tmp/ipykernel_23/50937330.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Starting Distillation on cuda...


Training:   0%|          | 0/4467 [00:00<?, ?it/s]/tmp/ipykernel_23/3678643907.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



--- Epoch 1/10 ---
Train Loss: 0.1465
Val Cosine Similarity: 0.9191 (Target: 1.0)
Val MSE: 0.0013
⭐ New Best Model Saved! (Similarity: 0.9191)



--- Epoch 2/10 ---
Train Loss: 0.0770
Val Cosine Similarity: 0.9377 (Target: 1.0)
Val MSE: 0.0010
⭐ New Best Model Saved! (Similarity: 0.9377)


...


--- Epoch 7/10 ---
Train Loss: 0.0421
Val Cosine Similarity: 0.9533 (Target: 1.0)
Val MSE: 0.0007
⭐ New Best Model Saved! (Similarity: 0.9533)



--- Epoch 8/10 ---
Train Loss: 0.0396
Val Cosine Similarity: 0.9530 (Target: 1.0)
Val MSE: 0.0007



--- Epoch 9/10 ---
Train Loss: 0.0374
Val Cosine Similarity: 0.9567 (Target: 1.0)
Val MSE: 0.0007
⭐ New Best Model Saved! (Similarity: 0.9567)



--- Epoch 10/10 ---
Train Loss: 0.0354
Val Cosine Similarity: 0.9546 (Target: 1.0)
Val MSE: 0.0007

Training Complete!


{'train_loss': [0.14645952902832687,
  0.07698628832282757,
  0.06262284710849855,
  0.05454572791307464,
  0.049233404060945375,
  0.04532699067648463,
  0.042138445800083156,
  0.039588438419953326,
  0.03740217271489213,
  0.03542287449951016],
 'val_cos': [0.9190794951037357,
  0.9377211646029824,
  0.943018304674249,
  0.943440147136387,
  0.9476463590797625,
  0.9522717485302373,
  0.953282034710834,
  0.9529551706816021,
  0.9566876339284998,
  0.954595496779994],
 'val_mse': [0.0012643836702122108,
  0.0009731075337377229,
  0.0008903397844589659,
  0.0008837483933587608,
  0.0008180262842583225,
  0.0007457547670990033,
  0.000729968988851301,
  0.0007350762132677789,
  0.0006767566351060706,
  0.0007094461202780765]}

## Thoughts
<font size='4'> With the successful distillation of BioViL features into MobileViT, we are ready to build a generative VLM. By appending a projection layer and a decoder, the model will be trained to synthesize medical reports from medical images. The upcoming phase involves fine-tuning a decoder on the reports to ensure clinical accuracy. Although we may investigate further distillation techniques later, constructing the full generative VLM is our current priority. </font>